In [2]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# 1. Chargement des données prétraitées
file_path = "/home/hado/Projets/Projet_vinci_adt/Data/processed/processed_data.npz"
data = np.load(file_path, allow_pickle=True)

X_train_final = data['X_train']
X_test_final = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

print(f"Dimensions chargées - Train: {X_train_final.shape}, Test: {X_test_final.shape}")

# 2. Instanciation du modèle de référence (Baseline)
baseline_model = LogisticRegression(
    class_weight='balanced', 
    max_iter=2000, 
    random_state=42,
    n_jobs=-1
)

# 3. Entraînement
print("\nEntraînement de la Baseline (Régression Logistique) en cours...")
baseline_model.fit(X_train_final, y_train)

# 4. Inférence sur le jeu de test
y_pred_baseline = baseline_model.predict(X_test_final)

# 5. Évaluation
print("\n--- Rapport de Classification (Baseline) ---")
print(classification_report(y_test, y_pred_baseline))

print("--- Matrice de Confusion ---")
print(confusion_matrix(y_test, y_pred_baseline))

Dimensions chargées - Train: (8239, 3429), Test: (2060, 3429)

Entraînement de la Baseline (Régression Logistique) en cours...


/home/hado/Projets/Projet_vinci_adt/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



--- Rapport de Classification (Baseline) ---
              precision    recall  f1-score   support

           1       0.93      0.94      0.94       870
           2       0.90      0.81      0.85      1040
           3       0.39      0.65      0.48       150

    accuracy                           0.85      2060
   macro avg       0.74      0.80      0.76      2060
weighted avg       0.88      0.85      0.86      2060

--- Matrice de Confusion ---
[[821  39  10]
 [ 58 839 143]
 [  2  51  97]]


In [3]:
import lightgbm as lgb
import time
from sklearn.metrics import classification_report, confusion_matrix

# 1. Ajustement des labels pour LightGBM
# LightGBM exige que les classes multiclasses commencent strictement à 0. 
# Nos classes actuelles sont 1, 2, 3. Nous devons les décaler à 0, 1, 2.
y_train_lgb = y_train - 1
y_test_lgb = y_test - 1

# 2. Instanciation de LightGBM
# class_weight='balanced' gère le déséquilibre
# n_jobs=-1 exploite tous les threads de ton processeur
lgb_model = lgb.LGBMClassifier(
    class_weight='balanced',
    random_state=42,
    n_estimators=200,
    n_jobs=-1
)

# 3. Entraînement avec mesure du temps
print("Entraînement de LightGBM en cours...")
start_time = time.time()
lgb_model.fit(X_train_final, y_train_lgb)
end_time = time.time()
print(f"Entraînement terminé en {end_time - start_time:.2f} secondes.")

# 4. Inférence sur le jeu de test
y_pred_lgb = lgb_model.predict(X_test_final)

# 5. Évaluation (en réajustant les labels à +1 pour l'affichage)
print("\n--- Rapport de Classification (LightGBM) ---")
print(classification_report(y_test, y_pred_lgb + 1))

print("--- Matrice de Confusion ---")
print(confusion_matrix(y_test, y_pred_lgb + 1))

Entraînement de LightGBM en cours...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008081 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 99857
[LightGBM] [Info] Number of data points in the train set: 8239, number of used features: 493
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Entraînement terminé en 3.52 secondes.

--- Rapport de Classification (LightGBM) ---
              precision    recall  f1-score   support

           1       0.94      0.96      0.95       870
           2       0.89      0.94      0.91      1040
           3       0.78      0.34      0.47       150

    accuracy                           0.91      2060
   macro avg       0.87      0.75      0.78      2060
weighted avg       0.90      0.91      0.90      2060

--- Matrice de Confusion ---
[[836  34   0]
 